In [5]:
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import DensityMatrix, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error


In [ ]:
def outer(psi):
    v = psi.reshape(-1, 1)
    return v @ v.conj().T

def trdist(rho, sigma):
    return np.sum(np.abs(np.linalg.svdvals(rho - sigma))) / 2

def sample_gcauchy(beta, M=1):
    thresh = (1 + np.sqrt(5)) / 2
    cs = []
    while len(cs) < M:
        c = scipy.stats.cauchy.rvs()
        u = scipy.stats.uniform.rvs()
        h = (1 + c**2) / (1 + (c / np.sqrt(2)) ** 4)
        if u < h / thresh:
            cs.append(c)
    return beta * (np.array(cs) if M > 1 else cs[0])

def ite_pauli_gadget(qc, anc, cbit, pauli, gamma_dtau, *, data_qubits=None):
    if isinstance(pauli, (list, tuple)):
        pauli = "".join(pauli)
    n = len(pauli)
    data_qubits = list(range(n)) if data_qubits is None else data_qubits
    supp = [i for i, p in enumerate(pauli) if p != "I"]
    if not supp or gamma_dtau == 0:
        return qc
    mag = abs(gamma_dtau)
    for i in supp:
        q = data_qubits[i]
        if pauli[i] == "X":
            qc.h(q)
        elif pauli[i] == "Y":
            qc.sdg(q); qc.h(q)
    chain = [data_qubits[i] for i in supp]
    for a, b in zip(chain[:-1], chain[1:]):
        qc.cx(a, b)
    parity_q = chain[-1]
    if gamma_dtau > 0:
        qc.x(parity_q)
    phi = 2 * np.arccos(np.exp(-2 * mag))
    qc.crx(phi, parity_q, anc)
    qc.measure(anc, cbit)
    qc.reset(anc)
    if gamma_dtau > 0:
        qc.x(parity_q)
    for a, b in zip(chain[-2::-1], chain[-1:0:-1]):
        qc.cx(a, b)
    for i in reversed(supp)
        q = data_qubits[i]
        if pauli[i] == "X":
            qc.h(q)
        elif pauli[i] == "Y":
            qc.h(q); qc.s(q)
    return qc

def rte_pauli_gadget(qc, P, theta, qubits=None):
    n = len(P)
    qubits = list(range(n)) if qubits is None else qubits
    supp = [i for i, p in enumerate(P) if p != "I"]
    if not supp:
        return qc
    for i in supp:
        q = qubits[i]
        if P[i] == "X":
            qc.h(q)
        elif P[i] == "Y":
            qc.sdg(q); qc.h(q)
    chain = [qubits[i] for i in supp]
    for a, b in zip(chain[:-1], chain[1:]):
        qc.cx(a, b)
    qc.rz(2 * theta, chain[-1])
    for a, b in zip(chain[-2::-1], chain[-1:0:-1]):
        qc.cx(a, b)
    for i in reversed(supp):
        q = qubits[i]
        if P[i] == "X":
            qc.h(q)
        elif P[i] == "Y":
            qc.h(q); qc.s(q)
    return qc

def trotter_pauli(terms, alpha, *, steps=1, order=2):
    norm = [(t, 1.0) if isinstance(t, str) else (t[0], float(t[1])) for t in terms]
    def S(x, o):
        if o == 1:
            return [(P, x * w) for P, w in norm]
        if o == 2:
            half = x / 2
            fwd = [(P, half * w) for P, w in norm]
            return fwd + list(reversed(fwd))
        p = 1 / (4 - 4 ** (1 / (o - 1)))
        return S(p * x, o - 2) + S((1 - 2 * p) * x, o - 2) + S(p * x, o - 2)
    return S(alpha / steps, order) * steps

def spin_chain_H_terms(n, J=1.0, g=1.0):
    Hs = []
    for i in range(n - 1):
        p = list("I" * n); p[i] = p[i + 1] = "Z"
        Hs.append(("".join(p), -J))
    for i in range(n):
        p = list("I" * n); p[i] = "X"
        Hs.append(("".join(p), -J * g))
    return Hs

def H_to_matrix(Hs):
    n = len(Hs[0][0])
    Hm = np.zeros((2**n, 2**n), dtype=complex)
    for t in Hs:
        P, w = (t, 1.0) if isinstance(t, str) else (t[0], float(t[1]))
        Hm += w * Operator.from_label("".join(P[::-1])).data
    return Hm

def make_noise_model(p):
    if p is None or p == 0:
        return None
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 1), ["x", "h", "s", "sdg", "rz", "rx"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cx", "crx"])
    return nm

def _is_zero_key(k):
    s = str(k).replace(" ", "")
    if s.startswith("0x"):
        return int(s, 16) == 0
    return bool(s) and set(s) <= {"0"}

def _extract_zero_branch_dm(res, tqc, label="rho", atol=1e-14):
    by_key = res.data(tqc)[label]
    ks = [k for k in by_key if _is_zero_key(k)]
    if not ks:
        return None, 0.0
    rho = np.asarray(getattr(by_key[ks[0]], "data", by_key[ks[0]]), dtype=complex)
    w = float(np.real(np.trace(rho)))
    return (None, 0.0) if w <= atol else (rho / w, w)


In [32]:
n = 8
g = 1.4
J = 1.0
betas = np.linspace(1e-3, 1.0, 10)
trotter = (10, 2)
rte_samples = 100
ite_shot_batch = 100
ite_max_shots = 1000
batch_size = 16
noise = 1e-4

Hs = spin_chain_H_terms(n, J=J, g=g)
Hmat = H_to_matrix(Hs)
w, V = np.linalg.eigh(Hmat)
ground_rho = outer(V[:, 0])

In [33]:
def build_ite_circuit(Hs, n, beta, *, trotter):
    sched = trotter_pauli(Hs, beta, steps=trotter[0], order=trotter[1])
    main, anc, creg = QuantumRegister(n), QuantumRegister(1), ClassicalRegister(len(sched))
    qc = QuantumCircuit(main, anc, creg)
    for i in range(n):
        qc.h(i)
    for j, (P, c) in enumerate(sched):
        ite_pauli_gadget(qc, anc[0], creg[j], P, gamma_dtau=c)
    qc.save_density_matrix(qubits=list(range(n)), label="rho", conditional=True, unnormalized=True)
    return qc

def run_ite_density(Hs, n, beta, *, shot_batch, max_shots, trotter, noise_model, pbar=None):
    qc = build_ite_circuit(Hs, n, beta, trotter=trotter)
    sim = AerSimulator(method="density_matrix", noise_model=noise_model)
    sim.set_options(max_parallel_threads=0, max_parallel_experiments=0, max_parallel_shots=0)
    tqc = transpile(qc, sim)
    used = 0
    while used < max_shots:
        m = min(shot_batch, max_shots - used)
        res = sim.run(tqc, shots=m).result()
        rho, w = _extract_zero_branch_dm(res, tqc, label="rho")
        used += m
        if pbar is not None:
            pbar.update(1)
        if rho is not None:
            return rho, w / m, used
    D = 2**n
    return np.zeros((D, D), dtype=complex), 0.0, used

def build_rte_circuit_from_rho(Hs, n, rho_in, t, *, trotter):
    qc = QuantumCircuit(n)
    qc.set_density_matrix(DensityMatrix(rho_in))
    for P, c in trotter_pauli(Hs, t, steps=trotter[0], order=trotter[1]):
        rte_pauli_gadget(qc, P, c)
    qc.save_density_matrix(label="rho_out")
    return qc

def run_rte_from_rho(Hs, n, beta, rho_ite, *, samples, trotter, noise_model, batch_size=16, pbar=None):
    sim = AerSimulator(method="density_matrix", noise_model=noise_model)
    sim.set_options(max_parallel_threads=0, max_parallel_experiments=0, max_parallel_shots=1)
    out = []
    for s in range(0, samples, batch_size):
        m = min(batch_size, samples - s)
        ts = [sample_gcauchy(beta) for _ in range(m)]
        circuits = [build_rte_circuit_from_rho(Hs, n, rho_ite, t, trotter=trotter) for t in ts]
        tqcs = transpile(circuits, sim)
        res = sim.run(tqcs, shots=1).result()
        for tqc in tqcs:
            out.append(np.asarray(res.data(tqc)["rho_out"].data, dtype=complex))
        if pbar is not None:
            pbar.update(1)
    return sum(out) / len(out)

def run_split_over_betas(Hs, n, betas, *, trotter, rte_samples, ite_shot_batch, ite_max_shots, noise_model, batch_size=16):
    D = 2**n
    rho_ite = np.zeros((len(betas), D, D), dtype=complex)
    rho_rite = np.zeros((len(betas), D, D), dtype=complex)
    p_ite = np.zeros(len(betas), dtype=float)
    n_ite_batches = (ite_max_shots + ite_shot_batch - 1) // ite_shot_batch
    n_rte_batches = (rte_samples + batch_size - 1) // batch_size
    for bi, beta in enumerate(tqdm(betas, desc="split sweep", unit="beta")):
        with tqdm(total=n_ite_batches + n_rte_batches, desc=f"beta {bi + 1}/{len(betas)}", unit="batch", leave=False) as pbar_beta:
            rho_i, p, used = run_ite_density(Hs, n, beta, shot_batch=ite_shot_batch, max_shots=ite_max_shots, trotter=trotter, noise_model=noise_model, pbar=pbar_beta)
            rho_ite[bi], p_ite[bi] = rho_i, p
            pbar_beta.set_postfix_str(f"p_ite={p:.4f}, shots={used}")
            print(f"beta {bi + 1}/{len(betas)} ITE postselection prob: {p:.4f} (shots used: {used})")
            rho_rite[bi] = run_rte_from_rho(Hs, n, beta, rho_i, samples=rte_samples, trotter=trotter, noise_model=noise_model, batch_size=batch_size, pbar=pbar_beta)
    return rho_ite, rho_rite, p_ite


In [ ]:
quick_noise = 1e-4
quick_ite_shot_batch = 100
quick_ite_max_shots = 1000
quick_rte_samples = 1000
quick_batch_size = 64

rho_ite_q, rho_rite_q, p_ite_q = run_split_over_betas(
    Hs,
    n,
    betas,
    trotter=trotter,
    rte_samples=quick_rte_samples,
    ite_shot_batch=quick_ite_shot_batch,
    ite_max_shots=quick_ite_max_shots,
    noise_model=make_noise_model(quick_noise),
    batch_size=quick_batch_size,
)

tr_i_q = np.array([trdist(rho_ite_q[i], ground_rho) for i in range(len(betas))])
tr_r_q = np.array([trdist(rho_rite_q[i], ground_rho) for i in range(len(betas))])

plt.figure(figsize=(7, 4))
plt.plot(betas, tr_i_q, "--x", color="tab:red", mfc="none", label=rf"ITE only (p={quick_noise:g})")
plt.plot(betas, tr_r_q, "-o", color="tab:blue", label=rf"split r-ITE (p={quick_noise:g})")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$\frac{1}{2}\|\rho-\rho_0\|_1$")
plt.title(f"Quick split check, n={n}, g={g}, trotter={trotter}")
plt.legend()
plt.tight_layout()
plt.show()

print("quick ITE postselection probability per beta:", np.round(p_ite_q, 4))

split sweep:   0%|          | 0/10 [00:00<?, ?beta/s]

beta 1/10:   0%|          | 0/17 [00:00<?, ?batch/s]

In [ ]:
rho_ite, rho_rite, p_ite = run_split_over_betas(
    Hs,
    n,
    betas,
    trotter=trotter,
    rte_samples=rte_samples,
    ite_shot_batch=ite_shot_batch,
    ite_max_shots=ite_max_shots,
    noise_model=make_noise_model(noise),
    batch_size=batch_size,
)

tr_i = np.array([trdist(rho_ite[i], ground_rho) for i in range(len(betas))])
tr_r = np.array([trdist(rho_rite[i], ground_rho) for i in range(len(betas))])

plt.figure(figsize=(7, 4))
plt.plot(betas, tr_i, "--x", color="tab:red", mfc="none", label="ITE only")
plt.plot(betas, tr_r, "-o", color="tab:blue", label="split r-ITE")
plt.xlabel(r"$\beta$")
plt.ylabel(r"$\frac{1}{2}\|\rho-\rho_0\|_1$")
plt.title(f"Split pipeline, n={n}, g={g}, trotter={trotter}")
plt.legend()
plt.tight_layout()
plt.show()

print("ITE postselection probability per beta:", np.round(p_ite, 4))
